# 11 — Figure 2/3 model selection, Monte Carlo, and w(z) reconstruction

This notebook uses the completed 100-seed experiment from Notebook 10 and the fixed datasets from Notebook 08. Every new analysis creates a separate folder under `results/11_fig23_mc_and_reconstruction/`.

## 1. Selection protocol, frozen before examining w(z)

**Independent result:** Use the 100 Notebook 10 seed-selected expressions with the public DESI covariance. On a fixed grid from z=0 to 2.33, require finite distance, first/second derivatives and w(z); D_M(0)=0 within 1e-8; positive and increasing D_M for z>0; and a positive Eq. (5) denominator factor. Keep expressions with original Notebook 10 loss no more than 1.5 times the minimum eligible loss. Select the lowest complexity, then lowest loss, then lowest seed. This is our independent protocol, not a claim about the authors' selection rule.

**Paper comparison:** Show the printed Eq. (12)/(13) without changing its constants. Also refit the family a z exp(b z) by full-covariance generalized least squares. Neither is eligible to replace the independent selection merely because its w(z) resembles LambdaCDM.

**Structure sensitivity:** Display the spread of central w(z) curves from all eligible seed-selected expressions. This is search variation on fixed data, not an observational confidence interval.

## 2. Reconstruction and uncertainty model

For D=D_M/r_d and gamma=Omega_m(100 r_d/c)^2 in consistent h-inverse-Mpc units, paper Eq. (5) is

$$w(z)=-1-\frac{2(1+z)D''+3\gamma(1+z)^3(D')^3}{3[D'-\gamma(1+z)^3(D')^3]}.$$

Refit numerical coefficients of the fixed selected structure by GLS with the full covariance C. With J=partial f/partial theta, draw coefficients from the **first-order GLS approximation** Cov(theta)=(J^T C^(-1) J)^(-1). One shared r_d value is drawn per realization; its first-order response rescales all SN-derived distances together and enters gamma. Omega_m=0.3 is fixed. PySR is not rerun for each draw.

Report all finite realizations and, separately, a conditional pole-free diagnostic removing grid-bracketed denominator crossings without replacement. The 1/2/3-sigma quantiles are **pointwise**, not global confidence bands. Coefficient bands are conditional on model structure; seed variation is shown separately.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'results/08_fig23_data_preparation/fig2_pantheon.npz').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from src.fig23_reconstruction import run_notebook11
UPSTREAM = (PROJECT_ROOT / 'results/10_fig23_seed_stability' /
            'fig23_seeds_100_20261004T071816359765Z_101dd054')
OUTPUT_ROOT = PROJECT_ROOT / 'results/11_fig23_mc_and_reconstruction'
print('Upstream:', UPSTREAM)

## 3. Run a new experiment or inspect the latest completed one

Inspection mode is the default, preventing an accidental long rerun. Set RUN_ANALYSIS=True to create a new folder; the command-line runner does that automatically.

In [ ]:
RUN_ANALYSIS = False
N_MC = 10000
MC_SEED = 20261011
if RUN_ANALYSIS:
    EXPERIMENT_DIR, OVERVIEW = run_notebook11(PROJECT_ROOT, UPSTREAM, n_mc=N_MC, mc_seed=MC_SEED)
else:
    completed = sorted(p for p in OUTPUT_ROOT.glob('fig23_mc_*')
                       if (p / 'experiment.json').is_file() and
                       json.loads((p / 'experiment.json').read_text()).get('status') == 'complete')
    if not completed:
        raise FileNotFoundError('No completed Notebook 11 experiment. Set RUN_ANALYSIS=True.')
    EXPERIMENT_DIR = completed[-1]
    OVERVIEW = pd.read_csv(EXPERIMENT_DIR / 'overview.csv')
print('Experiment:', EXPERIMENT_DIR)
display(OVERVIEW)

## 4. Check selection and provenance

The candidate-diagnostics table contains every public-covariance selected expression and its physical checks. The scenario manifest records the chosen seed, original and refitted loss, parameter covariance, and source hashes.

In [ ]:
for scenario in ('fig2_pantheon', 'fig3_desy5'):
    meta = json.loads((EXPERIMENT_DIR / scenario / 'scenario.json').read_text())
    checks = pd.read_csv(EXPERIMENT_DIR / scenario / 'candidate_diagnostics.csv')
    print('\n', scenario, '| selected seed:', meta['selection']['selected_seed'])
    print('Raw independent expression:', meta['raw_selected_expression'])
    print('Eligible:', meta['selection']['eligible_count'], 'loss-gate finalists:', meta['selection']['finalist_count'])
    display(checks.loc[checks.independent_primary, ['seed', 'complexity', 'python_loss',
                   'sympy_expression', 'eligible', 'within_loss_gate']])

## 5. Inspect Monte Carlo diagnostics

The all-draw and pole-free intervals answer different questions. Always report how many draws remain after pole filtering. Three-sigma pointwise tails may be noisy when few draws survive.

In [ ]:
for scenario in ('fig2_pantheon', 'fig3_desy5'):
    meta = json.loads((EXPERIMENT_DIR / scenario / 'scenario.json').read_text())
    print('\n', scenario)
    for branch, details in meta['branches'].items():
        diagnostic = details['diagnostics']
        print(branch, 'loss:', round(details['loss'], 4), 'finite:', diagnostic['n_finite'],
              'pole-free:', diagnostic['n_pole_free'])
    band = pd.read_csv(EXPERIMENT_DIR / scenario / 'independent_bands.csv')
    display(band.loc[(band.quantity == 'w') & (band.selection == 'all')]
            .iloc[::60][['z', 'q00135', 'q025', 'median', 'q975', 'q99865']])

## 6. View the reconstructions

Colored shading shows pointwise observational 95% coefficient-propagation bands conditional on each structure. Gray shading is the eligible-seed structural spread, **not** a confidence interval. Display limits clip extreme w(z) values; exported CSV files retain the full quantiles.

In [ ]:
for scenario in ('fig2_pantheon', 'fig3_desy5'):
    display(Image(filename=str(EXPERIMENT_DIR / scenario / 'figure_reconstruction.png')))

## 7. Interpretation guardrails

- A low distance loss does not guarantee stable w(z): the reconstruction uses first and second derivatives.
- Values exactly at z=0 are extrapolations; the lowest SN fit redshift is above zero.
- The exponential comparison cannot establish why the authors selected their expression.
- Do not choose a formula because it agrees with or contradicts LambdaCDM.
- A pointwise exclusion of w=-1 under one chosen expression is not a global model rejection. Report sensitivity to expression choice, r_d, redshift range, and pole treatment.
- The Notebook 10 published-loss discrepancy remains a separate, unresolved audit question.